# 03 — BIO Tagging for NER on CTI Text

**Goal:** Before we fine-tune any NER model, we need to understand how NER labels are actually represented. You'll learn the **BIO tagging scheme**, label CTI sentences by hand, and see why this specific format is used.

## Why this notebook exists

In notebook 2 we saw that NER gives **one label per token**. But what are those labels?

A naive approach would be: "just label each word with its entity type, or `O` if it's not an entity." That sounds fine until two named entities appear next to each other — e.g., `Emotet Cobalt Strike`. Without extra info, a reader (or a model) can't tell whether that's one entity or two.

**BIO** solves this by marking where each entity *begins* vs *continues*.

## What you'll learn

1. Picking an entity schema for CTI (what types of entity we care about)
2. The BIO tagging format: `B-TYPE`, `I-TYPE`, `O`
3. Why BIO beats a flat label set, with a concrete adjacent-entity example
4. Hand-labeling a CTI sentence
5. Variants: BIOES / IOBES — when and why people use them
6. The label-to-ID mapping (`label2id`, `id2label`) that models need

## Step 1 — Pick an entity schema for CTI

Before labeling anything, you must decide **what entity types you care about**. For cyber threat intelligence, common types include:

| Type | Description | Examples |
|---|---|---|
| `THREAT_ACTOR` | Adversary group or person | APT28, Lazarus Group, Fancy Bear |
| `MALWARE` | Malware family or tool | Emotet, X-Agent, Cobalt Strike |
| `VULNERABILITY` | CVE IDs, named flaws | CVE-2021-44228, Log4Shell |
| `TOOL` | Legitimate tool used offensively | PowerShell, mimikatz |
| `ORG` | Targeted organization | Sony Pictures, Equifax |
| `LOC` | Geographic location | Ukraine, Eastern Europe |
| `IOC` | Indicator of compromise | IPs, hashes, domains |

For this notebook we'll use a small, focused schema — just enough to be interesting:

- `THREAT_ACTOR`
- `MALWARE`
- `VULNERABILITY`

In real projects, the schema is driven by what you want to do **downstream** (build an IOC search engine? track actor activity? map to MITRE ATT&CK?). Start small — adding labels later is easy; reducing them is painful because you have to re-label.

> Existing CTI datasets use schemas that already exist. **DNRTI** has ~13 entity types; **MalwareTextDB** uses a small set; **STIX** defines many more. Borrowing an existing schema saves you from reinventing the wheel.

## Step 2 — The BIO tagging scheme

BIO stands for **Begin / Inside / Outside**. Given a token sequence, each token gets one label:

- `B-TYPE` — this token is the **first** token of an entity of type `TYPE`.
- `I-TYPE` — this token is **inside** (a continuation of) an entity of type `TYPE`.
- `O` — this token is **outside** any entity.

So with 3 entity types, our label set is:
```
O
B-THREAT_ACTOR, I-THREAT_ACTOR
B-MALWARE,      I-MALWARE
B-VULNERABILITY, I-VULNERABILITY
```
= **7 labels total** (one `O` + 2 per entity type).

**Rule:** `I-X` must be preceded by `B-X` or another `I-X` of the same type. An isolated `I-X` with no `B-X` before it is malformed — your model can still predict it, but good evaluation libraries (like `seqeval`) will flag it.

## Step 3 — Why BIO beats flat labels (adjacent-entity problem)

Consider this fragment:

> `... deployed Emotet Cobalt Strike beacons ...`

Here `Emotet` and `Cobalt Strike` are **two separate malware entities** that happen to appear next to each other.

### Naive flat labels

| Token    | Label    |
|----------|----------|
| deployed | O        |
| Emotet   | MALWARE  |
| Cobalt   | MALWARE  |
| Strike   | MALWARE  |
| beacons  | O        |

**Problem:** Is this *one* 3-token malware entity (`Emotet Cobalt Strike`) or *two* separate ones (`Emotet` and `Cobalt Strike`)? You can't tell from the labels.

### BIO labels

| Token    | Label       |
|----------|-------------|
| deployed | O           |
| Emotet   | B-MALWARE   |
| Cobalt   | B-MALWARE   |  ← a new `B-` = new entity starts here
| Strike   | I-MALWARE   |
| beacons  | O           |

Now it's unambiguous: two entities, not one. **This is the whole reason BIO exists.**

## Step 4 — Hand-label a CTI sentence (word level)

Let's do it for real. We'll work at the **word level first** (no subwords yet) — that's how human annotators work, and it's easier to reason about. In notebook 4 we'll handle the subword alignment when we feed this to BERT.

Our sentence: *"APT28 deployed X-Agent malware exploiting CVE-2017-0199 last March."*

Expected entities:
- `APT28` → `THREAT_ACTOR`
- `X-Agent` → `MALWARE`
- `CVE-2017-0199` → `VULNERABILITY`

In [1]:
# A simple word-level representation: list of (word, label) pairs.
# In real datasets this is typically stored in CoNLL format (word \t label per line).

labeled_sentence = [
    ("APT28",        "B-THREAT_ACTOR"),
    ("deployed",     "O"),
    ("X-Agent",      "B-MALWARE"),
    ("malware",      "O"),              # the word "malware" itself isn't part of the entity name
    ("exploiting",   "O"),
    ("CVE-2017-0199","B-VULNERABILITY"),
    ("last",         "O"),
    ("March",        "O"),              # we decided not to include DATE in our schema
    (".",            "O"),
]

# Pretty print
print(f"{'Word':<16} {'Label':<18}")
print("-" * 36)
for word, label in labeled_sentence:
    print(f"{word:<16} {label:<18}")

Word             Label             
------------------------------------
APT28            B-THREAT_ACTOR    
deployed         O                 
X-Agent          B-MALWARE         
malware          O                 
exploiting       O                 
CVE-2017-0199    B-VULNERABILITY   
last             O                 
March            O                 
.                O                 


### Things to notice

- **Every token gets a label.** There's no such thing as "no label" — non-entities are explicitly `O`.
- **Single-token entities use `B-` only.** `APT28` is one word, so only `B-THREAT_ACTOR` — no `I-` follows.
- **Words like "malware"** that describe an entity but aren't part of the entity name stay `O`. This is a judgment call your annotation guidelines must make explicit.
- **Entity boundaries depend on your schema.** We chose not to label dates, so `last March` is `O O`. If we later added a `DATE` type, we'd go back and re-label.

> **Annotation guidelines matter more than you'd think.** Two annotators will disagree on edge cases ("is 'the Emotet gang' an actor or a malware mention?"). For serious projects, write a 1-2 page guidelines doc with examples before labeling.

## Step 5 — A multi-word entity example

Now for a sentence with a **multi-token** entity, so you see `I-` in action.

*"The Lazarus Group deployed Cobalt Strike beacons targeting Sony Pictures."*

- `Lazarus Group` → `THREAT_ACTOR` (2 tokens)
- `Cobalt Strike` → `MALWARE` (2 tokens)
- `Sony Pictures` → we'd label as `ORG` if it were in our schema — but it isn't, so `O`

In [2]:
labeled_sentence_2 = [
    ("The",       "O"),
    ("Lazarus",   "B-THREAT_ACTOR"),
    ("Group",     "I-THREAT_ACTOR"),   # continuation of same entity
    ("deployed",  "O"),
    ("Cobalt",    "B-MALWARE"),
    ("Strike",    "I-MALWARE"),        # continuation
    ("beacons",   "O"),
    ("targeting", "O"),
    ("Sony",      "O"),                 # would be B-ORG if we had ORG in our schema
    ("Pictures",  "O"),                 # would be I-ORG
    (".",         "O"),
]

print(f"{'Word':<12} {'Label':<18}")
print("-" * 32)
for word, label in labeled_sentence_2:
    print(f"{word:<12} {label:<18}")

Word         Label             
--------------------------------
The          O                 
Lazarus      B-THREAT_ACTOR    
Group        I-THREAT_ACTOR    
deployed     O                 
Cobalt       B-MALWARE         
Strike       I-MALWARE         
beacons      O                 
targeting    O                 
Sony         O                 
Pictures     O                 
.            O                 


## Step 6 — Recovering entity spans from BIO labels

A BIO sequence is a label per token, but for *using* predictions you usually want entity **spans** like `("Lazarus Group", "THREAT_ACTOR", start=1, end=2)`.

Here's a simple function that groups consecutive `B-X`, `I-X`, `I-X` tokens back into spans.

> In production you'd use `seqeval.metrics` for metrics and `transformers.pipelines.token_classification` has its own aggregation — but writing it once yourself clarifies what these libraries are actually doing.

In [3]:
def extract_entities(labeled):
    """Given a list of (word, BIO-label) pairs, return list of (text, type, start, end) spans.
    start/end are token indices, inclusive."""
    spans = []
    current = None  # (type, start_idx, words)

    for i, (word, label) in enumerate(labeled):
        if label == "O":
            if current is not None:
                spans.append(current + (i - 1,))  # close previous span
                current = None
            continue

        prefix, etype = label.split("-", 1)  # "B"/"I", "THREAT_ACTOR"

        if prefix == "B":
            # close any running entity, then start a new one
            if current is not None:
                spans.append(current + (i - 1,))
            current = (etype, i, [word])
        elif prefix == "I":
            # continuation — extend the current entity if types match
            if current is not None and current[0] == etype:
                current[2].append(word)
            else:
                # malformed: I-X with no preceding B-X of same type -> treat as new entity
                current = (etype, i, [word])

    # close any still-open entity at end of sentence
    if current is not None:
        spans.append(current + (len(labeled) - 1,))

    # reshape into readable dicts
    return [
        {"text": " ".join(words), "type": etype, "start": start, "end": end}
        for (etype, start, words), end in [((s[0], s[1], s[2]), s[3]) for s in spans]
    ]


print("Entities from sentence 1:")
for ent in extract_entities(labeled_sentence):
    print(" ", ent)

print("\nEntities from sentence 2:")
for ent in extract_entities(labeled_sentence_2):
    print(" ", ent)

Entities from sentence 1:
  {'text': 'APT28', 'type': 'THREAT_ACTOR', 'start': 0, 'end': 0}
  {'text': 'X-Agent', 'type': 'MALWARE', 'start': 2, 'end': 2}
  {'text': 'CVE-2017-0199', 'type': 'VULNERABILITY', 'start': 5, 'end': 5}

Entities from sentence 2:
  {'text': 'Lazarus Group', 'type': 'THREAT_ACTOR', 'start': 1, 'end': 2}
  {'text': 'Cobalt Strike', 'type': 'MALWARE', 'start': 4, 'end': 5}


## Step 7 — The adjacent-entity test

Let's verify that BIO really handles the case BIO was designed for. We'll construct the `Emotet Cobalt Strike beacons` example and extract the entities.

In [4]:
adjacent = [
    ("The",      "O"),
    ("operator", "O"),
    ("loaded",   "O"),
    ("Emotet",   "B-MALWARE"),
    ("Cobalt",   "B-MALWARE"),   # <-- new B- means a new entity starts
    ("Strike",   "I-MALWARE"),
    ("beacons",  "O"),
    (".",        "O"),
]

print("Entities extracted:")
for ent in extract_entities(adjacent):
    print(" ", ent)

Entities extracted:
  {'text': 'Emotet', 'type': 'MALWARE', 'start': 3, 'end': 3}
  {'text': 'Cobalt Strike', 'type': 'MALWARE', 'start': 4, 'end': 5}


You should see **two** separate MALWARE entities: `Emotet` and `Cobalt Strike`. That's BIO earning its keep.

## Step 8 — Variants: BIOES (also called IOBES)

BIO is the most common scheme but has subtle ambiguities. **BIOES** adds two more tags:

- `B-X` — **beginning** of a multi-token entity
- `I-X` — **inside** (middle) of a multi-token entity
- `E-X` — **end** of a multi-token entity
- `S-X` — **single-token** entity (S for "single")
- `O` — outside

Our earlier sentence in BIOES:

| Word | BIO | BIOES |
|---|---|---|
| APT28 | B-THREAT_ACTOR | **S**-THREAT_ACTOR |
| deployed | O | O |
| X-Agent | B-MALWARE | **S**-MALWARE |
| malware | O | O |
| Lazarus | B-THREAT_ACTOR | B-THREAT_ACTOR |
| Group | I-THREAT_ACTOR | **E**-THREAT_ACTOR |

**Pros of BIOES:** the model gets a stronger training signal — it learns an explicit "end" concept. On some datasets this yields ~0.5–1 F1 point improvement.

**Cons:** twice as many labels (`B/I/E/S` × N types + O), more label-imbalance, more code.

**Recommendation:** start with BIO. It's the Hugging Face default and matches almost every public dataset. Switch to BIOES only if you've measured and specifically need the boost.

## Step 9 — `label2id` and `id2label`

Models don't speak strings — they predict integer class IDs. Before training, you build two dictionaries:

- `label2id`: string → integer (used when **labeling** training data)
- `id2label`: integer → string (used when **decoding** model predictions)

Hugging Face models store these on `model.config`, so they travel with saved checkpoints. That's why if you load a fine-tuned NER model, it already knows the label names.

In [5]:
# Build the full BIO label set for our CTI schema
entity_types = ["THREAT_ACTOR", "MALWARE", "VULNERABILITY"]

labels = ["O"]
for etype in entity_types:
    labels.append(f"B-{etype}")
    labels.append(f"I-{etype}")

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

print("Total labels:", len(labels))
print()
print("label2id:")
for label, idx in label2id.items():
    print(f"  {label:<18} -> {idx}")

Total labels: 7

label2id:
  O                  -> 0
  B-THREAT_ACTOR     -> 1
  I-THREAT_ACTOR     -> 2
  B-MALWARE          -> 3
  I-MALWARE          -> 4
  B-VULNERABILITY    -> 5
  I-VULNERABILITY    -> 6


### Convert our hand-labeled sentence to integer IDs

This is the format a model would consume during training (well, almost — we still have subword alignment ahead, which is the next notebook).

In [6]:
words = [w for w, _ in labeled_sentence]
label_ids = [label2id[l] for _, l in labeled_sentence]

print(f"{'Word':<16} {'Label':<18} {'ID'}")
print("-" * 44)
for w, (_, lbl), lid in zip(words, labeled_sentence, label_ids):
    print(f"{w:<16} {lbl:<18} {lid}")

print("\nwords:", words)
print("label_ids:", label_ids)

Word             Label              ID
--------------------------------------------
APT28            B-THREAT_ACTOR     1
deployed         O                  0
X-Agent          B-MALWARE          3
malware          O                  0
exploiting       O                  0
CVE-2017-0199    B-VULNERABILITY    5
last             O                  0
March            O                  0
.                O                  0

words: ['APT28', 'deployed', 'X-Agent', 'malware', 'exploiting', 'CVE-2017-0199', 'last', 'March', '.']
label_ids: [1, 0, 3, 0, 0, 5, 0, 0, 0]


## Your turn — exercises

1. **Label by hand:** write a function-returning list of `(word, BIO-label)` pairs for this sentence:
   > *"The Lazarus Group used Cobalt Strike and Mimikatz against targets running Log4j, exploiting CVE-2021-44228."*
   Then run `extract_entities(...)` on it and confirm you recover the 4 entities you expected.

2. **Adjacent entities of different types:** construct a sentence where a `THREAT_ACTOR` is immediately followed by a `MALWARE` (no words between them), label it, and verify the extraction correctly identifies both.

3. **Break the schema:** what happens if you put `I-MALWARE` at the start of a sentence, with no `B-MALWARE` before it? Run `extract_entities` on it and look at what comes out. (This is *malformed* output that real models sometimes emit — your span-extraction code should degrade gracefully.)

4. **BIO → BIOES conversion:** write a short function that converts `labeled_sentence_2` from BIO to BIOES.

In [ ]:
# Your experiments here


## Summary

- NER labels are **one per token**, not one per entity.
- **BIO** uses `B-TYPE` for entity starts, `I-TYPE` for continuations, `O` for non-entities.
- BIO disambiguates **adjacent entities of the same type** — the key scenario a flat scheme can't handle.
- For 3 entity types you need 7 labels (1 × `O` + 2 × 3).
- `label2id` / `id2label` connect the human-readable labels to the integer classes the model predicts.
- **BIOES** is a richer variant — usually not worth the extra complexity unless you've measured it helps.

## Next notebook

**`04_subword_label_alignment.ipynb`** — we have word-level labels, but BERT works in subwords. A `B-MALWARE` label on `X-Agent` has to become labels on the 3 subword tokens (`x`, `-`, `agent`). You'll learn the standard Hugging Face approach using `is_split_into_words=True` and `word_ids()`. This is the last puzzle piece before we actually fine-tune an NER model.